This notebook builds and evaluates multiple classifiers on an imbalanced binary dataset, applies SMOTE for balancing, and logs all models and metrics to an MLflow tracking server. It showcases:
- Data imbalance handling
- Model comparison
- MLflow experiment tracking
- Signature and input example logging for reproducibility


Load essential libraries for data generation, modeling, evaluation, and MLflow tracking.

In [13]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report
import warnings
warnings.filterwarnings('ignore')


Simulate a binary classification problem with class imbalance (90% class 0, 10% class 1). This reflects real-world fraud or anomaly detection scenarios

In [14]:
# Step 1: Create an imbalanced binary classification dataset
X, y = make_classification(n_samples=1000, n_features=10, n_informative=2, n_redundant=8, 
                           weights=[0.9, 0.1], flip_y=0, random_state=42)

np.unique(y, return_counts=True)

(array([0, 1]), array([900, 100]))

Split data while preserving class distribution. Stratification ensures both sets reflect the imbalance

In [15]:
# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)

In [16]:
log_reg = LogisticRegression(C=1, solver='liblinear')
log_reg.fit(X_train, y_train)
y_pred_log_reg = log_reg.predict(X_test)
print(classification_report(y_test, y_pred_log_reg))

              precision    recall  f1-score   support

           0       0.95      0.96      0.95       270
           1       0.60      0.50      0.55        30

    accuracy                           0.92       300
   macro avg       0.77      0.73      0.75       300
weighted avg       0.91      0.92      0.91       300



- Compare three classifiers—Logistic Regression, Random Forest, and XGBoost—on the imbalanced dataset. 
- Evaluation uses precision, recall, and F1-score to highlight performance on minority class.

In [17]:
rf_clf = RandomForestClassifier(n_estimators=30, max_depth=3)
rf_clf.fit(X_train, y_train)
y_pred_rf = rf_clf.predict(X_test)
print(classification_report(y_test, y_pred_rf))

              precision    recall  f1-score   support

           0       0.97      1.00      0.98       270
           1       0.95      0.70      0.81        30

    accuracy                           0.97       300
   macro avg       0.96      0.85      0.89       300
weighted avg       0.97      0.97      0.96       300



In [18]:
xgb_clf = XGBClassifier(use_label_encoder=False, eval_metric='logloss')
xgb_clf.fit(X_train, y_train)
y_pred_xgb = xgb_clf.predict(X_test)
print(classification_report(y_test, y_pred_xgb))

              precision    recall  f1-score   support

           0       0.98      1.00      0.99       270
           1       0.96      0.80      0.87        30

    accuracy                           0.98       300
   macro avg       0.97      0.90      0.93       300
weighted avg       0.98      0.98      0.98       300



Balance the training data using SMOTE (oversampling minority) + Tomek links (undersampling noisy majority). This improves recall and F1-score for class 1.

In [19]:
from imblearn.combine import SMOTETomek

smt = SMOTETomek(random_state=42)
X_train_res, y_train_res = smt.fit_resample(X_train, y_train)

np.unique(y_train_res, return_counts=True)

(array([0, 1]), array([619, 619]))

In [20]:
# Evaluate how balancing affects model performance. You’ll notice improved recall and F1-score for class 1.

xgb_clf = XGBClassifier(use_label_encoder=False, eval_metric='logloss')
xgb_clf.fit(X_train_res, y_train_res)
y_pred_xgb = xgb_clf.predict(X_test)
print(classification_report(y_test, y_pred_xgb))

              precision    recall  f1-score   support

           0       0.98      0.98      0.98       270
           1       0.81      0.83      0.82        30

    accuracy                           0.96       300
   macro avg       0.89      0.91      0.90       300
weighted avg       0.96      0.96      0.96       300



In [21]:
# Organize models and their training/testing data into a list for loop-based training and logging.
models = [
    (
        "Logistic Regression", 
        LogisticRegression(C=1, solver='liblinear'), 
        (X_train, y_train),
        (X_test, y_test)
    ),
    (
        "Random Forest", 
        RandomForestClassifier(n_estimators=30, max_depth=3), 
        (X_train, y_train),
        (X_test, y_test)
    ),
    (
        "XGBClassifier",
        XGBClassifier(use_label_encoder=False, eval_metric='logloss'), 
        (X_train, y_train),
        (X_test, y_test)
    ),
    (
        "XGBClassifier With SMOTE",
        XGBClassifier(use_label_encoder=False, eval_metric='logloss'), 
        (X_train_res, y_train_res),
        (X_test, y_test)
    )
]

In [22]:
reports = []

for model_name, model, train_set, test_set in models:
    X_train = train_set[0]
    y_train = train_set[1]
    X_test = test_set[0]
    y_test = test_set[1]
    
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    report = classification_report(y_test, y_pred, output_dict=True)
    reports.append(report)

Automate training and evaluation for each model. Stores metrics in a structured format for MLflow logging
Track each model’s parameters, metrics, and artifacts in MLflow. Includes:
- input_example for reproducibility
- signature for schema validation
- Separate handling for XGBoost vs scikit-learn models

In [23]:
import mlflow
import mlflow.sklearn
import mlflow.xgboost
from mlflow.models import infer_signature

mlflow.set_tracking_uri("http://127.0.0.1:5001")
experiment_name = "Anomaly Detection"

# Create or get experiment ID
if mlflow.get_experiment_by_name(experiment_name) is None:
    experiment_id = mlflow.create_experiment(experiment_name)
else:
    experiment_id = mlflow.get_experiment_by_name(experiment_name).experiment_id

# Log each model
for i, element in enumerate(models):
    model_name = element[0]
    model = element[1]
    report = reports[i]

    input_example = X_test[:1]
    signature = infer_signature(X_test, model.predict(X_test))

    with mlflow.start_run(run_name=model_name, experiment_id=experiment_id):
        mlflow.log_param("model", model_name)
        mlflow.log_metric("accuracy", report["accuracy"])
        mlflow.log_metric("recall_class_1", report["1"]["recall"])
        mlflow.log_metric("recall_class_0", report["0"]["recall"])
        mlflow.log_metric("f1_score_macro", report["macro avg"]["f1-score"])

        if "XGB" in model_name:
            mlflow.xgboost.log_model(model, name="model", input_example=input_example, signature=signature)
        else:
            mlflow.sklearn.log_model(model, name="model", input_example=input_example, signature=signature)

🏃 View run Logistic Regression at: http://127.0.0.1:5001/#/experiments/5/runs/1b450c021e934782b866ad1a508aee48
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/5
🏃 View run Random Forest at: http://127.0.0.1:5001/#/experiments/5/runs/f908ce8487624a938028929fdc0fec1a
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/5
🏃 View run XGBClassifier at: http://127.0.0.1:5001/#/experiments/5/runs/1f9d9fa2a14546c4b1ecd62ffd23de82
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/5
🏃 View run XGBClassifier With SMOTE at: http://127.0.0.1:5001/#/experiments/5/runs/f46f7badb5ba4417a97fb99a5e04cd72
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/5
